# 3. GPT-2 training and test inference


In [ ]:
!pip install accelerate -U
!pip install transformers[torch]

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    IntervalStrategy,
    LineByLineTextDataset,
    TrainingArguments,
    Trainer,
)

In [ ]:
import pandas as pd

In [ ]:
KAGGLE_TRAIN = False

if KAGGLE_TRAIN:
    !wget https://media.githubusercontent.com/media/danielpancake/text-detox/main/data/interim/processed.tsv

    train_data_file = "/kaggle/working/processed.tsv"
    model_output_dir = "/kaggle/working/models/gpt2-based"
else:
    train_data_file = "../data/interim/processed.tsv"
    model_output_dir = "../models/gpt2-based"

In [ ]:
df = pd.read_csv(train_data_file, sep="\t", header=None, names=["tox", "detox"])
df.head()

In [ ]:
# Add used tokens:
# [TOX], [/TOX], [DETOX], [/DETOX], »»
tokens_dict = {
    "tox_begin": "[TOX]",
    "tox_end": "[/TOX]",
    "detox_begin": "[DETOX]",
    "detox_end": "[/DETOX]",
    "separator": "»»",
}

In [ ]:
# Make a combined column of the toxic and detoxified sentences with the special tokens
df["combined"] = (
    tokens_dict["tox_begin"]
    + df["tox"]
    + tokens_dict["tox_end"]
    + tokens_dict["separator"]
    + tokens_dict["detox_begin"]
    + df["detox"]
    + tokens_dict["detox_end"]
)

# Save the combined sentences to a text file
df["combined"].to_csv("combined.txt", index=False, header=False)

df.head()

In [ ]:
batch_size = 8
epochs = 1

# For the block size, we use a length of the longest sentence in our dataset
block_size = df["combined"].map(len).max() + 1

warmup_steps = 500
save_steps = 10_000
logging_steps = 500

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2", cache_dir="cache")
tokenizer = AutoTokenizer.from_pretrained("gpt2", cache_dir="cache")
datacollator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

In [ ]:
num_added_toks = tokenizer.add_tokens(list(tokens_dict.values()))
print(f"Added {num_added_toks} tokens")
model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

In [ ]:
train_dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path="combined.txt",
    block_size=block_size
)

In [ ]:
training_args = TrainingArguments(
    output_dir=model_output_dir,
    overwrite_output_dir=True,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    warmup_steps=warmup_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=datacollator,
    train_dataset=train_dataset,
)

trainer.train()
trainer.save_model(model_output_dir)

In [ ]:
# Inference pipeline
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device="cuda")

In [ ]:
def detoxify(text: str, params: dict = None) -> str:
    prompt = (
        tokens_dict["tox_begin"]
        + text
        + tokens_dict["tox_end"]
        + tokens_dict["separator"]
        + tokens_dict["detox_begin"]
    )
    max_length = len(prompt) * 2.5
    return generator(prompt, max_length=max_length)

In [ ]:
prompt = "I hate your stupid fucking face!!"

detoxify(prompt)